# 02b · Entrenamiento — U-Net++ (CNN mejorada)

Entrena **U-Net++** sobre los datos de Kaggle .

U-Net++ añade conexiones densas entre encoder y decoder que mejoran
la captura de estructuras finas — ideal para capas retinales delgadas.

| Variante | Fuentes | Clases |
|----------|---------|--------|
| v1_kaggle_8c | Kaggle | 9 (8+fondo) |

**Metricas:** Dice · IoU · F1 · Hausdorff
**Salida:** checkpoints/unetpp/{variant}/ + logs/unetpp_{variant}.csv

In [1]:
# ============================================================
#  CABECERA DE ENTORNO
# ============================================================
import sys, os
from pathlib import Path

def _detect_env() -> str:
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'): return 'kaggle'
    try:
        import google.colab; return 'colab'
    except ImportError: pass
    try:
        get_ipython(); return 'notebook'
    except NameError: return 'script'

ENV = _detect_env()
IS_NOTEBOOK = ENV in ('colab', 'kaggle', 'notebook')
print(f'[ENV] {ENV}')

if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT_DIR = Path('/content/drive/MyDrive/oct-retina-segmentation')
elif ENV == 'kaggle':
    ROOT_DIR = Path('/kaggle/input/datasets/brbaracerezo/data-unified')
    ROOT_OUT = Path('/kaggle/working')
else:
    _here = Path(globals().get('__file__', './')).resolve()
    ROOT_DIR = _here.parents[1] if _here.suffix == '.py' else _here.parent
    ROOT_OUT = _here.parents[1] if _here.suffix == '.py' else _here.parent

DATA_DIR     = ROOT_DIR /'VC2'/ 'data'
VARIANTS_DIR = DATA_DIR / 'Data_Kaggle'
CKPT_DIR     = ROOT_OUT / 'checkpoints' / 'unetpp'
LOGS_DIR     = ROOT_OUT / 'logs'
ALL_RESULTS = ROOT_OUT / 'results'  
ALL_RESULTS_PATH = ROOT_OUT / 'all_results.json'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)
print(f'[ENV] VARIANTS_DIR = {VARIANTS_DIR}')
print(f'[ENV] CKPT_DIR     = {CKPT_DIR}')
print(f'[ENV] ALL_RESULTS = {ALL_RESULTS}')

[ENV] notebook
[ENV] VARIANTS_DIR = C:\Users\User\oct-retina-segmentation\OCT\data\Data_Kaggle
[ENV] CKPT_DIR     = C:\Users\User\oct-retina-segmentation\OCT\checkpoints\unetpp
[ENV] ALL_RESULTS = C:\Users\User\oct-retina-segmentation\OCT\results


In [2]:
# ============================================================
#  IMPORTS
# ============================================================
import json, csv, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import KFold
from scipy.ndimage import distance_transform_edt
import cv2
import shutil


try:
    from tqdm.notebook import tqdm as tqdm_fn
except ImportError:
    from tqdm import tqdm as tqdm_fn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[OK] Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'     GPU : {torch.cuda.get_device_name(0)}')
    print(f'     VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

[OK] Device: cpu


In [3]:
MODEL_NAME = 'unetpp'

N_FOLDS             = 2
N_FOLDS_OCT         = 3
NUM_EPOCHS          = 30
NUM_EPOCHS_OCT      = 20
BATCH_SIZE          = 6      # UNet++ usa mas memoria por las conexiones densas
LR                  = 1e-4
WEIGHT_DECAY        = 1e-5
EARLY_STOP_PATIENCE = 10
IMG_SIZE            = (256, 256)
IMG_SIZE_OCT        = (256, 256)  # en lugar de mantener 640×400
NUM_WORKERS         = 2
SEED                = 42

USE_WANDB     = False
WANDB_PROJECT = 'oct-retina-segmentation'

DEBUG_MODE = False #True   # cambia a False para produccion
if DEBUG_MODE:
    NUM_EPOCHS = 2
    N_FOLDS    = 2
    print('[DEBUG] 2 epocas, 2 folds.')

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'[OK] {MODEL_NAME}: epochs={NUM_EPOCHS}  folds={N_FOLDS}  batch={BATCH_SIZE}  lr={LR}')


[OK] unetpp: epochs=30  folds=2  batch=6  lr=0.0001


---
## Bloque 1 · Dataset

In [4]:
class VariantDataset(Dataset):
    """
    Carga una variante pre-procesada (.npy) para PyTorch.
    - Resize a IMG_SIZE
    - Normalizacion [0,1] con clipping de percentiles
    - Augmentation: flip horizontal aleatorio
    """
    def __init__(self, variant_name, variants_dir,
                 img_size=(256,256), augment=False, clip_pct=(1.0, 99.0)):
        self.img_size = img_size
        self.augment  = augment
        self.clip_pct = clip_pct

        imgs_path = variants_dir / 'resized_images.npy'
        lbls_path = variants_dir / 'resized_labeledimages.npy'
        assert imgs_path.exists(), f'No encontrado: {imgs_path}'
        print(f'[imgs_path:] {imgs_path}')

        # mmap_mode='r' → lee del disco bajo demanda, no carga todo en RAM
        self.images = np.load(imgs_path, mmap_mode='r')
        self.labels = np.load(lbls_path, mmap_mode='r')

    def __len__(self): return len(self.images)

    def _normalize(self, img):
        p_lo, p_hi = np.percentile(img, self.clip_pct)
        img = np.clip(img, p_lo, p_hi)
        return ((img - p_lo) / max(p_hi - p_lo, 1e-8)).astype(np.float32)

    def __getitem__(self, idx):
        img  = self.images[idx].astype(np.float32)
        mask = self.labels[idx].astype(np.float32)
        H, W = self.img_size
        img  = cv2.resize(img,  (W, H), interpolation=cv2.INTER_AREA)
        mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)
        img  = self._normalize(img)
        if self.augment and np.random.rand() > 0.5:
            img  = np.fliplr(img).copy()
            mask = np.fliplr(mask).copy()
        return {
            'image': torch.from_numpy(img).unsqueeze(0).float(),
            'mask':  torch.from_numpy(mask).long(),
        }

print('[OK] VariantDataset.')

[OK] VariantDataset.


---
## Bloque 2 · Modelo U-Net++

Conexiones densas entre encoder y decoder. Params: ~36M.

In [5]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UpConcat(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x_up, *skip_list):
        x = self.up(x_up)
        x = torch.cat([x, *skip_list], dim=1)
        return self.conv(x)

class UNetPP(nn.Module):
    """
    U-Net++ con conexiones densas en el decoder.
    Entrada:  [B, 1, H, W]
    Salida:   [B, n_classes, H, W]
    Params:   ~36M
    """
    def __init__(self, n_channels=1, n_classes=9):
        super().__init__()
        f = [64, 128, 256, 512, 1024]

        # Encoder
        self.enc0 = DoubleConv(n_channels, f[0])
        self.enc1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(f[0], f[1]))
        self.enc2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(f[1], f[2]))
        self.enc3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(f[2], f[3]))
        self.enc4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(f[3], f[4]))

        # Nodos intermedios densos
        self.x01 = UpConcat(f[1] + f[0],        f[0])
        self.x11 = UpConcat(f[2] + f[1],        f[1])
        self.x21 = UpConcat(f[3] + f[2],        f[2])
        self.x31 = UpConcat(f[4] + f[3],        f[3])

        self.x02 = UpConcat(f[1] + f[0]*2,      f[0])
        self.x12 = UpConcat(f[2] + f[1]*2,      f[1])
        self.x22 = UpConcat(f[3] + f[2]*2,      f[2])

        self.x03 = UpConcat(f[1] + f[0]*3,      f[0])
        self.x13 = UpConcat(f[2] + f[1]*3,      f[1])

        self.x04 = UpConcat(f[1] + f[0]*4,      f[0])

        self.outc = nn.Conv2d(f[0], n_classes, 1)

    def forward(self, x):
        x00 = self.enc0(x)
        x10 = self.enc1(x00)
        x20 = self.enc2(x10)
        x30 = self.enc3(x20)
        x40 = self.enc4(x30)

        x01 = self.x01(x10, x00)
        x11 = self.x11(x20, x10)
        x21 = self.x21(x30, x20)
        x31 = self.x31(x40, x30)

        x02 = self.x02(x11, x00, x01)
        x12 = self.x12(x21, x10, x11)
        x22 = self.x22(x31, x20, x21)

        x03 = self.x03(x12, x00, x01, x02)
        x13 = self.x13(x22, x10, x11, x12)

        x04 = self.x04(x13, x00, x01, x02, x03)

        return self.outc(x04)

# Verificacion
_m = UNetPP(1, 9)
_x = torch.randn(2, 1, 256, 256)
_y = _m(_x)
assert _y.shape == (2, 9, 256, 256)
params = sum(p.numel() for p in _m.parameters())
print(f'[OK] UNet++  output={_y.shape}  params={params/1e6:.1f}M')
del _m, _x, _y


[OK] UNet++  output=torch.Size([2, 9, 256, 256])  params=36.6M


---
## Bloque 3 · Loss y métricas

In [6]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        n = logits.shape[1]
        p = F.softmax(logits, dim=1)
        t = F.one_hot(targets, n).permute(0,3,1,2).float()
        dims  = (0, 2, 3)
        inter = (p * t).sum(dims)
        union = p.sum(dims) + t.sum(dims)
        return 1 - ((2*inter + self.smooth) / (union + self.smooth)).mean()

class CombinedLoss(nn.Module):
    """0.5 * CrossEntropy + 0.5 * Dice — robusto para segmentacion multiclase."""
    def __init__(self):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss()
        self.dice = DiceLoss()
    def forward(self, logits, targets):
        return 0.5 * self.ce(logits, targets) + 0.5 * self.dice(logits, targets)

def compute_metrics(preds, targets, n_classes, hausdorff=False):
    """
    preds, targets : np.ndarray [N, H, W] int
    Retorna dict con Dice/IoU/F1/Hausdorff por clase y promedio.
    Hausdorff es costoso — activar solo en la ultima epoca.
    """
    dice_list, iou_list, hd_list = [], [], []

    for c in range(n_classes):
        p = (preds == c)
        t = (targets == c)
        inter = (p & t).sum()
        union = (p | t).sum()
        dice  = 2 * inter / (p.sum() + t.sum() + 1e-8)
        iou   = inter / (union + 1e-8)
        dice_list.append(float(dice))
        iou_list.append(float(iou))

        if hausdorff and p.any() and t.any():
            hd_vals = []
            for i in range(len(preds)):
                pi, ti = (preds[i]==c), (targets[i]==c)
                if pi.any() and ti.any():
                    d1 = distance_transform_edt(~ti)[pi].max()
                    d2 = distance_transform_edt(~pi)[ti].max()
                    hd_vals.append(max(d1, d2))
            hd_list.append(float(np.mean(hd_vals)) if hd_vals else 0.)
        else:
            hd_list.append(0.)

    return {
        'dice_per_class': dice_list,
        'iou_per_class':  iou_list,
        'hd_per_class':   hd_list,
        'dice_mean': float(np.mean(dice_list)),
        'iou_mean':  float(np.mean(iou_list)),
        'f1_mean':   float(np.mean(dice_list)),   # F1 = Dice en multiclase
        'hd_mean':   float(np.mean(hd_list)),
    }

print('[OK] DiceLoss + CombinedLoss + compute_metrics.')

[OK] DiceLoss + CombinedLoss + compute_metrics.


---
## Bloque 4 · Loop de entrenamiento

In [7]:
class CSVLogger:
    """Agrega filas a un CSV — crea cabecera automaticamente si no existe."""
    def __init__(self, path): self.path = Path(path)
    def log(self, row: dict):
        write_header = not self.path.exists()
        with open(self.path, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=row.keys())
            if write_header: w.writeheader()
            w.writerow(row)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total = 0.
    for batch in loader:
        imgs  = batch['image'].to(device)
        masks = batch['mask'].to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / max(len(loader), 1)

def validate_one_epoch(model, loader, criterion, device, n_classes, compute_hd=False):
    model.eval()
    total = 0.
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            imgs  = batch['image'].to(device)
            masks = batch['mask'].to(device)
            out   = model(imgs)
            total += criterion(out, masks).item()
            all_preds.append(torch.argmax(out, dim=1).cpu().numpy())
            all_targets.append(masks.cpu().numpy())
    preds   = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)
    metrics = compute_metrics(preds, targets, n_classes, hausdorff=compute_hd)
    return total / max(len(loader), 1), metrics

print('[OK] train_one_epoch / validate_one_epoch / CSVLogger.')

[OK] train_one_epoch / validate_one_epoch / CSVLogger.


In [8]:
def run_kfold(variant_name, n_classes,
              n_folds=N_FOLDS,
              num_epochs=NUM_EPOCHS,
              img_size=IMG_SIZE):        
    """
    Ejecuta 5-fold CV de U-Net sobre una variante.
    Guarda:
      CKPT_DIR/{variant}/fold_K_best.pth   — mejor por fold
      CKPT_DIR/{variant}/best_overall.pth  — mejor global
      LOGS_DIR/unet_{variant}.csv          — metricas por epoca
    """    
    print(f'\n{"="*60}')
    print(f'  U-Net++  |  {variant_name}  |  {n_classes} clases')
    print(f'  folds={n_folds}  epochs={num_epochs}  img_size={img_size} ')
    print(f'{"="*60}')

    ds        = VariantDataset(variant_name, VARIANTS_DIR,
                               img_size=img_size, augment=True)
    logger    = CSVLogger(LOGS_DIR / f'unetpp_{variant_name}.csv')
    ckpt_base = CKPT_DIR / variant_name
    ckpt_base.mkdir(parents=True, exist_ok=True)

    # wandb opcional
    wb_run = None
    if USE_WANDB:
        try:
            import wandb
            wb_run = wandb.init(
                project=WANDB_PROJECT,
                name=f'unetpp__{variant_name}',
                config={'variant': variant_name, 'model': 'unetpp',
                        'n_classes': n_classes, 'epochs': num_epochs,
                        'folds': n_folds, 'lr': LR, 'batch': BATCH_SIZE},
                reinit=True,
            )
        except Exception as e:
            print(f'  [WARN] wandb: {e}')

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    fold_results      = []
    best_overall_dice = -1.
    
    # Verificar que el resize está aplicándose
    sample = ds[0] 
    print(f'Shape real del tensor: {sample["image"].shape}')  
    # debe mostrar torch.Size([1, 256, 256])
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(range(len(ds)))):
        print(f'\n  --- Fold {fold+1}/{n_folds} '
              f'(train={len(train_idx)}  val={len(val_idx)}) ---')

        train_loader = DataLoader(
                Subset(ds, train_idx), batch_size=BATCH_SIZE,
                shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
            )

        val_loader = DataLoader(
            Subset(ds, val_idx), batch_size=BATCH_SIZE,
            shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
        )

        model     = UNetPP(n_channels=1, n_classes=n_classes).to(DEVICE)
        criterion = CombinedLoss()
        optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

        best_fold_loss    = float('inf')
        best_fold_metrics = {}
        best_fold_dice    = -1.
        patience_ctr      = 0

        for epoch in range(num_epochs):
            t0         = time.time()
            train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
            # Hausdorff solo en ultima epoca (muy costoso)            
            compute_hd = (epoch == num_epochs - 1)
            val_loss, val_m = validate_one_epoch(
                model, val_loader, criterion, DEVICE, n_classes, compute_hd
            )
            scheduler.step()
            lr_now = optimizer.param_groups[0]['lr']

            print(f'    Ep {epoch+1:3d}/{num_epochs}  '
                  f'train={train_loss:.4f}  val={val_loss:.4f}  '
                  f'dice={val_m["dice_mean"]:.4f}  '
                  f'iou={val_m["iou_mean"]:.4f}  '
                  f'lr={lr_now:.2e}  t={time.time()-t0:.1f}s')
            # CSV
            row = {
                'model': 'unetpp', 'variant': variant_name,
                'fold': fold+1, 'epoch': epoch+1,
                'train_loss': round(train_loss, 6),
                'val_loss':   round(val_loss,   6),
                'dice': round(val_m['dice_mean'], 6),
                'iou':  round(val_m['iou_mean'],  6),
                'f1':   round(val_m['f1_mean'],   6),
                'hd':   round(val_m['hd_mean'], 4) if compute_hd else '',
                'lr':   round(lr_now, 8),
            }
            logger.log(row)
            if wb_run:
                wb_run.log({**row, 'step': epoch + fold * num_epochs})
            # Checkpoint por fold
            if val_loss < best_fold_loss:
                best_fold_loss    = val_loss
                best_fold_dice    = val_m['dice_mean']
                best_fold_metrics = val_m
                patience_ctr      = 0
                torch.save({
                    'fold': fold+1, 'epoch': epoch+1,
                    'model_state_dict': model.state_dict(),
                    'val_loss': val_loss, 'metrics': val_m,
                    'variant': variant_name, 'n_classes': n_classes,
                }, ckpt_base / f'fold_{fold+1}_best.pth')
            else:
                patience_ctr += 1

            if patience_ctr >= EARLY_STOP_PATIENCE:
                print(f'    Early stopping en ep {epoch+1}'); break
        # Checkpoint global
        if best_fold_dice > best_overall_dice:
            best_overall_dice = best_fold_dice
            torch.save({
                'fold': fold+1,
                'model_state_dict': model.state_dict(),
                'val_loss': best_fold_loss,
                'metrics': best_fold_metrics,
                'variant': variant_name, 'n_classes': n_classes,
            }, ckpt_base / 'best_overall.pth')
            print(f'  * Nuevo best_overall: dice={best_overall_dice:.4f} (fold {fold+1})')

        fold_results.append({
            'fold': fold+1, 'val_loss': best_fold_loss, **best_fold_metrics
        })
    # Resumen del CV
    dices = [r['dice_mean'] for r in fold_results]
    ious  = [r['iou_mean']  for r in fold_results]
    hds   = [r['hd_mean']   for r in fold_results]
    print(f'\n  RESUMEN U-Net++ / {variant_name}:')
    print(f'  Dice : {np.mean(dices):.4f} +/- {np.std(dices):.4f}')
    print(f'  IoU  : {np.mean(ious):.4f} +/- {np.std(ious):.4f}')
    print(f'  HD   : {np.mean(hds):.2f}')

    # START BACKUP    
    for pth in (CKPT_DIR / 'unetpp').rglob('*.pth'):
        dest = ROOT_OUT / 'ckpt_backup' / pth.relative_to(CKPT_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(pth, dest)
        print(f'  Backup: {pth.name}')
    for csv_f in LOGS_DIR.glob('unetpp_*.csv'):
        shutil.copy2(csv_f, ROOT_OUT / csv_f.name)
        print(f'  Backup: {csv_f.name}')
    print(f'[OK] Backup {variant_name} completado.')
    # END BACKUP    
    if wb_run:
        wb_run.summary.update({'dice_cv_mean': np.mean(dices),
                               'iou_cv_mean':  np.mean(ious)})
        wb_run.finish()

    return fold_results

print('[OK] run_kfold.')

[OK] run_kfold.


## Bloque 5 · Guardar resultados

In [9]:
import json

def guardar_all_results(all_results, path):
    """Guarda ALL_RESULTS a disco en JSON."""
    # Convertir numpy types a Python nativos para JSON
    def convert(obj):
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        if isinstance(obj, dict):        return {k: convert(v) for k, v in obj.items()}
        if isinstance(obj, list):        return [convert(v) for v in obj]
        return obj

    with open(path, 'w') as f:
        json.dump(convert(all_results), f, indent=2)
    print(f'[OK] ALL_RESULTS guardado: {path}')

def cargar_all_results(path):
    """Carga ALL_RESULTS desde disco."""
    if not Path(path).exists():
        print(f'[WARN] No encontrado: {path}')
        return {}
    with open(path) as f:
        return json.load(f)
    print(f'[OK] ALL_RESULTS cargado: {path}')

In [10]:
# Acumulador de resultados para la tabla final
ALL_RESULTS = {}
print('[OK] ALL_RESULTS inicializado.')

# Al inicio del notebook — cargar si ya existe
ALL_RESULTS = cargar_all_results(ALL_RESULTS_PATH)

[OK] ALL_RESULTS inicializado.


---
## Bloque 6 · Entrenamiento — una celda por variante

Corré solo las celdas que necesitás.  
Cada celda es **independiente** — podés relanzar una sola si falla.

In [ ]:
# ----------------------------------------------------------
# V1 · Solo Kaggle · 8 capas (9 clases con fondo)
# ----------------------------------------------------------
try:
    ALL_RESULTS['v1_kaggle_8c'] = run_kfold('v1_kaggle_8c', n_classes=9,
                                           n_folds=N_FOLDS,num_epochs=NUM_EPOCHS,img_size=IMG_SIZE)
except Exception as e:
    print(f'[ERROR] v1_kaggle_8c: {e}')
    ALL_RESULTS['v1_kaggle_8c'] = None
finally:
    guardar_all_results(ALL_RESULTS, ALL_RESULTS_PATH)  # ← guarda siempre  




  U-Net++  |  v1_kaggle_8c  |  9 clases
  folds=2  epochs=30  img_size=(256, 256) 
[imgs_path:] C:\Users\User\oct-retina-segmentation\OCT\data\Data_Kaggle\resized_images.npy
Shape real del tensor: torch.Size([1, 256, 256])

  --- Fold 1/2 (train=110  val=110) ---
